# Weakly-supervised Video Anomaly Detection with Robust Temporal Feature Magnitude Learning
**Tian, Pang, Chen, Singh, Verjans, Carneiro. ICCV 2021**

## Overview

Weakly-Supervised Video Anomaly Detection (WSVAD) assumes that only video-level labels are available during training. A video is labeled as either **normal** or **abnormal**, but the exact temporal location of the anomaly is unknown. For example, a surveillance video may be labeled as anomalous because it contains a robbery, even though the robbery only occurs during a few seconds of a much longer video.

Most WSVAD methods are based on the **Multiple Instance Learning (MIL)** framework. In MIL, a video is divided into multiple temporal segments (*snippets*). A normal video forms a **negative bag**, where all snippets are expected to be normal. An anomalous video forms a **positive bag**, where at least one snippet is anomalous, although the model does not know which one.

## Problem with Traditional MIL Methods

A common strategy is to assign an anomaly score to every snippet and select the snippet with the highest score as evidence of the anomaly.

Example:

| Snippet | Anomaly Score |
|----------|--------------|
| 1 | 0.20 |
| 2 | 0.95 |
| 3 | 0.60 |
| 4 | 0.70 |
| 5 | 0.10 |

MIL selects **Snippet 2** because it has the highest anomaly score.

However, suppose the actual situation is:

- Snippet 1: Normal
- Snippet 2: Normal
- Snippet 3: Anomaly
- Snippet 4: Anomaly
- Snippet 5: Normal

In this case, the model incorrectly treats Snippet 2 as anomalous. As a result, the network learns from a normal snippet while believing it is observing an anomaly. This introduces **label noise** into the positive bag and can reinforce incorrect patterns during training.

This is one of the main limitations of traditional MIL-based approaches.

## Feature Magnitude

RTFM argues that anomaly scores alone are not sufficiently reliable for identifying anomalous snippets. Instead, it focuses on the **feature magnitude** produced by a deep feature extractor.

Each snippet is represented by a feature vector. For example:

**Normal snippet**

```text
f = [1, 2]
```

**Anomalous snippet**

```text
f = [5, 6]
```

The magnitude of a feature vector is computed using its L2 norm:

```text
||f|| = √(f₁² + f₂² + ... + fₙ²)
```

Applying this to the examples:

```text
||[1,2]|| = √(1² + 2²)
          = √5
          ≈ 2.24
```

```text
||[5,6]|| = √(5² + 6²)
          = √61
          ≈ 7.81
```

The anomalous snippet has a significantly larger feature magnitude.

## Intuition Behind RTFM

The key observation made by the authors is that anomalous snippets tend to generate feature vectors with larger magnitudes than normal snippets.

For example:

**Normal event (walking)**

```text
[0.2, 0.3, 0.1, 0.4, ...]
```

**Abnormal event (fighting or robbery)**

```text
[2.5, 3.1, 1.8, 2.9, ...]
```

The abnormal event produces stronger activations in the feature space, resulting in a larger magnitude.

Geometrically, feature vectors can be viewed as points in a high-dimensional space:

```text
       *
    *
  *
O
```

```text
        X

      X


O
```

Where:

- `O` = origin
- `X` = anomalous snippets
- `*` = normal snippets

Normal snippets tend to remain closer to the origin, while anomalous snippets are located farther away. The distance from the origin corresponds to the feature magnitude.

## Main Contribution

The main contribution of RTFM is to explicitly learn a representation where anomalous snippets have larger feature magnitudes than normal snippets.

Instead of relying solely on anomaly scores:

```text
Video
  ↓
Top-scoring snippet
  ↓
Anomaly prediction
```

RTFM introduces feature magnitude as an additional cue:

```text
Video
  ↓
Feature extraction
  ↓
Feature magnitude
  ↓
More reliable anomaly localization
```

By encouraging anomalous snippets to have larger magnitudes than normal snippets, the model becomes more robust to the label noise introduced by incorrect snippet selection in traditional MIL methods.

## Key Takeaway

Traditional MIL-based methods may incorrectly select normal snippets as evidence of anomalies, creating noisy supervision signals. RTFM addresses this problem by learning a feature space in which anomalous snippets naturally exhibit larger feature magnitudes than normal snippets, allowing more reliable anomaly localization and improving weakly-supervised video anomaly detection performance.

## Results

<div>
    <img src='../images/RTFM_T1.png' width="600">
</div>

---

<div>
    <img src='../images/RTFM_T2.png' width="600">
</div>

---

<div>
    <img src='../images/RTFM_T3.png' width="600">
</div>

---

<div>
    <img src='../images/RTFM_T4.png' width="600">
</div>

---

We reduce the number of abnormal training videos from the original 63 videos down to 25 videos, with the normal training videos and test data fixed. As expected, the performance of both our method and Sultani decreases with decreasing number of abnormal training videos, but the decreasing rate of our model is smaller that of than Sultani. This is because RTFM performs better recognition of the positive instances in the abnormal videos, and as a result, it can leverage the same training data more effectively than a MIL-based approach.

<div>
    <img src='../images/RTFM_EFFICIENCY.png' width="600">
</div>

---

<div>
    <img src='../images/RFTM_EXAMPLES.png' width="1200">
</div>